# Stage 2 (revision 2) — price differential privacy on the channel that is actually open

## What the first run of Part A found

| | |
|---|---|
| RW-MER, full 12-template matrix | **0.0046 — Low**, against 0.3985 for the same records at 60 exposures, **with no defense applied** |
| Held-out competence | **0.427** novel-and-valid vs the released model's 0.280 (z = 3.76, p = 1.7×10⁻⁴) |
| Held-out **recitation** | **0.227** vs **0.000** for the base weights |

The first two lines read as a clean win, and that is how the first version of
this notebook's gate scored it: RW-MER is Low, so stop.

That gate was reading the wrong channel. **RW-MER is defined on person-keyed
probes** — all 11,760 queries in the matrix ask about a *training* subject — so
it measures **attributed** disclosure and is structurally unable to see a model
that emits a real training identifier under someone else's name. The held-out
probe does see that, and it ran at 0.227 against exactly zero for the base
weights. **The channel is entirely induced by fine-tuning, and it is invisible
to the benchmark's own headline metric.**

So there *is* something for a privacy budget to remove. Part B is back on.

## Three changes in this revision

**1. Records are kept.** Every condition writes `records_matrix_*.csv` (11,760
rows) and `records_heldout_*.csv`. The first run returned aggregates only, which
meant the question "where did the 74 surviving hits land?" could be answered
only by an analytic upper bound — *even in the most concentrated case the worst
per-category MER is under 0.09* — and not by measurement. Aggregates are not
enough to audit an aggregate.

**2. Part B is scored on recitation, not RW-MER.** The aggregate is already Low;
moving it further is not evidence of anything. A budget earns its keep here by
closing the unattributed channel while keeping competence.

**3. The held-out probe is 180 people (n = 900), not 60 (n = 300).** The span to
be priced is 0.280 → 0.427, i.e. 0.147. At n = 300 the standard error is 0.029,
so "keeps half the span" sits about 2.5 standard errors from "keeps none" — too
close to call. At n = 900 the error drops to 0.017.

## Part A has to be re-run

Not because it was wrong, but because it did not keep its records, and because
the held-out probe is now three times larger. About 80 minutes. Part B is gated
on it as before, on the revised two-channel test.

---

**Before starting:** Runtime → Change runtime type → **L4 GPU**.
Part A ≈ 80 min. Part B ≈ 95 min per budget, four budgets — plan on two
sessions. Everything checkpoints per condition; re-running a cell resumes.

## 1 — Confirm the GPU

In [ ]:
!nvidia-smi

## 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/piibench'
os.makedirs(DRIVE, exist_ok=True)
print('results will be written to:', DRIVE)

## 3 — Unpack the bundle

Upload `piibench_epsilon.zip` into the **`piibench` folder** of your Drive first, then run this cell.

If you ran the previous revision, delete `epsilon_140/epsilon_sweep.json` from Drive first — its rows were measured on a 300-probe held-out set and are not comparable with the 900-probe one used here.

In [ ]:
import zipfile, os
src = f'{DRIVE}/piibench_epsilon.zip'
assert os.path.exists(src), f'not found: {src} - upload the zip to Drive first'
zipfile.ZipFile(src).extractall('/content/work')
os.chdir('/content/work')
print(sorted(os.listdir('.')))

## 4 — Install dependencies
(about 2 minutes)

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets pandas

## 5 — Part A: audit the generalization window, and keep the rows this time

Fine-tunes at four exposures on the varied corpus (140 records, matching Chapters
5 and 6), then runs the complete twelve-template audit plus a 900-probe held-out
measurement, writing every query to CSV.

**About 80 minutes on an L4.** Read the GATE block at the end before continuing.

In [ ]:
!python epsilon_sweep.py --part A \
  --model Qwen/Qwen2.5-1.5B \
  --persons 140 --epochs 2 --repeats 2 \
  --heldout-persons 180 \
  --out-dir $DRIVE/epsilon_140 \
  --adapter-dir /content/adapters_eps

## 6 — Read the gate

This also prints the category breakdown of the surviving hits — the table the first run could not produce.

In [ ]:
import json
res = json.load(open(f'{DRIVE}/epsilon_140/epsilon_sweep.json'))
u, b = res['undefended'], res['base']
print(f"  ATTRIBUTED    RW-MER {u['rwmer']:.4f} -> {u['band']}"
      f"   ({u['raw_hits']} raw hits in {u['n_probes']} probes)")
print(f"  UNATTRIBUTED  recitation {u['recitation']:.4f}  vs base {b['recitation']:.4f}")
print(f"  COMPETENCE    {b['novel_valid']:.3f} -> {u['novel_valid']:.3f}"
      f"   (span {u['novel_valid']-b['novel_valid']:+.3f}, n={u['n_heldout']})")
print()
print('  where the surviving hits sit:')
cats = u['matrix']['by_category']
for c, e in sorted(cats.items(), key=lambda x: -x[1]['hits']):
    if e['hits']:
        pri = u['matrix']['per_category_metric'][c]['pri']
        print(f"    {c:<22} {e['hits']:>4}/{e['n']}  rate {e['rate']:.4f}  PRI {pri}")
print()
leaks = (u['band'] != 'Low') or (u['recitation'] - b['recitation'] > 0.02)
print('RUN CELL 7' if leaks else 'SKIP CELL 7 - neither channel leaks; go to cell 9')

## 7 — Part B: the epsilon sweep

Run only if cell 6 said to. Four budgets, **about 95 minutes each** — DP-SGD
computes one backward pass per example in order to clip per-example gradients,
so training dominates even at four exposures.

**The budgets are ordered 1.0 → 4.0 → 0.5 → 2.0 on purpose.** That is a coarse
bracket first: if the session dies after two conditions you already know whether
the usable point lies at the loose end, the tight end, or in between, instead of
having four adjacent points at one end of the curve.

In [ ]:
!python epsilon_sweep.py --part B \
  --model Qwen/Qwen2.5-1.5B \
  --persons 140 --epochs 2 --repeats 2 \
  --heldout-persons 180 \
  --epsilons 1.0 4.0 0.5 2.0 \
  --noise-multipliers 0.342 0.098 0.648 0.182 \
  --out-dir $DRIVE/epsilon_140 \
  --adapter-dir /content/adapters_eps

## 8 — Read the curve

In [ ]:
import json
res = json.load(open(f'{DRIVE}/epsilon_140/epsilon_sweep.json'))
b, u = res.get('base'), res.get('undefended')
span = (u['novel_valid'] - b['novel_valid']) if (b and u) else 0.0
leak = (u['recitation'] - b['recitation']) if (b and u) else 0.0
print('  PRIMARY axis is recitation. RW-MER was already Low before any defense.\n')
print(f"  {'condition':<22} {'eps':>7} {'recite':>8} {'RW-MER':>8} {'novel':>8} {'kept':>7} {'closed':>8}")
print(f"  {'base':<22} {'-':>7} {b['recitation']:>8.3f} {0.0:>8.4f} {b['novel_valid']:>8.3f} {'-':>7} {'-':>8}")
print(f"  {'undefended':<22} {'-':>7} {u['recitation']:>8.3f} {u['rwmer']:>8.4f} {u['novel_valid']:>8.3f} {'100%':>7} {'0%':>8}")
best = None
for k in sorted([k for k in res if k.startswith('dp_')], key=lambda k: res[k]['epsilon']):
    r = res[k]
    kept = (r['novel_valid'] - b['novel_valid']) / span if span > 1e-9 else 0.0
    closed = (u['recitation'] - r['recitation']) / leak if leak > 1e-9 else 0.0
    print(f"  {k:<22} {r['epsilon']:>7.3f} {r['recitation']:>8.3f} {r['rwmer']:>8.4f} "
          f"{r['novel_valid']:>8.3f} {kept*100:>6.0f}% {closed*100:>7.0f}%")
    if closed >= 0.8 and kept >= 0.5 and (best is None or r['epsilon'] < res[best]['epsilon']):
        best = k
print()
if best:
    r = res[best]
    print(f"  USABLE BUDGET: {best}, eps {r['epsilon']:.3f}")
    print(f"  Closes the recitation channel and keeps most of the competence span.")
    print(f"  This is the first priced privacy-utility point in the study.")
else:
    print('  NO USABLE BUDGET among those swept. Say which axis failed:')
    print('   - closes the leak but competence falls to base level -> repeats the')
    print('     Section 6.5.1 finding at a better operating point (strengthens it)')
    print('   - keeps competence but recitation survives -> the noise never reached')
    print('     the channel that carries it (a different, also reportable, result)')
se = (0.25 / u['n_heldout']) ** 0.5
print(f"\n  n = {u['n_heldout']}, SE about {se:.3f}; span {span:.3f} is {span/se:.1f} SE.")

## 9 — Collect the results

Now several MB rather than a few KB, because the per-query records travel with the summary. The cell asserts they are present.

In [ ]:
import shutil, os, glob
os.makedirs('/content/send', exist_ok=True)
src = f'{DRIVE}/epsilon_140'
for pat in ['epsilon_sweep.json', 'dataset.json', 'records_*.csv']:
    for f in glob.glob(f'{src}/{pat}'):
        shutil.copy(f, '/content/send/')
sent = sorted(os.listdir('/content/send'))
for f in sent:
    print(f"  {f:<40} {os.path.getsize('/content/send/'+f)/1024:>9.1f} KB")
shutil.make_archive(f'{DRIVE}/epsilon_results', 'zip', '/content/send')
sz = os.path.getsize(f'{DRIVE}/epsilon_results.zip')
print(f"\n  {len(sent)} files -> {DRIVE}/epsilon_results.zip  ({sz/1e6:.1f} MB)")
assert any(f.startswith('records_matrix') for f in sent), \
    'no matrix records - the per-query rows are the point of this revision'
print('  records present. Download from Drive and send it back.')

---
### If the session drops
Re-run cells 2 → 3 → 4, then the cell you were on. Completed conditions are detected and skipped, so Part B resumes at the next budget.

### If you hit CUDA out of memory
Add `--batch-size 12` to the command.

### What good output looks like
Part A prints audit progress with a rate and ETA, then a two-channel GATE block. Part B prints one block per budget ending in `-> eps X  RW-MER Y band  novel_valid Z`.

### If Part B refuses to run
It now refuses only when **both** channels are clean. Given that Part A already measured recitation at 0.227, that should not happen — if it does, something changed upstream and the JSON is worth sending back before anything else.